# Medicare+ — Prescription Handwriting OCR
## Notebook 1 of 3: Data Preparation

**Goal:** Download the Kaggle Doctor's Handwritten Prescription dataset, explore it, split into train/validation/test, and save as a HuggingFace `Dataset` ready for fine-tuning TrOCR.

**Run this in Google Colab.** Runtime > Change runtime type > GPU (T4) is recommended.

### Outline
1. Install dependencies
2. Mount Google Drive (for persistence)
3. Authenticate to Kaggle
4. Download & extract the dataset
5. Explore: image counts, label distribution, image dimensions, sample visualization
6. Build a clean `(image_path, label)` table
7. Train/val/test split (80/10/10, stratified by drug class where possible)
8. Save splits to Drive as a HuggingFace `DatasetDict`

### Dataset

We use [`mamun1113/doctors-handwritten-prescription-bd-dataset`](https://www.kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset) — ~4,700 cropped images of single drug names handwritten by doctors, with the printed name as label.

> **Note:** This dataset contains *cropped single-word* images, not full prescription sheets. The model trained on it will recognize one drug name per image. Full-page processing is handled by a text-detection step (CRAFT) in the backend pipeline at inference time.

## 1. Install dependencies

In [ ]:
!pip install -q kaggle datasets pandas pillow scikit-learn matplotlib

## 2. Mount Google Drive

We'll save the prepared dataset to Drive so notebooks 2 and 3 can reuse it without re-downloading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/medicare_plus_ocr'
DATA_ROOT = f'{PROJECT_ROOT}/data'
os.makedirs(DATA_ROOT, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

## 3. Authenticate to Kaggle

Get your Kaggle API token: https://www.kaggle.com/settings → "Create New API Token" → downloads `kaggle.json`.

Upload it below (run this cell, click "Choose Files", select `kaggle.json`).

In [ ]:
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials in place.')

## 4. Download & extract the dataset

In [ ]:
KAGGLE_DATASET = 'mamun1113/doctors-handwritten-prescription-bd-dataset'
RAW_DIR = f'{DATA_ROOT}/raw'
os.makedirs(RAW_DIR, exist_ok=True)

if not os.listdir(RAW_DIR):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {RAW_DIR} --unzip
else:
    print('Dataset already downloaded — skipping.')

!ls -lah {RAW_DIR} | head -20

## 5. Discover the dataset structure

Different versions of the Kaggle dataset organize files slightly differently (sometimes `Training/Word/<class>/*.png`, sometimes a CSV index). Let's auto-detect.

In [ ]:
from pathlib import Path
import pandas as pd

raw = Path(RAW_DIR)

csv_files = list(raw.rglob('*.csv'))
image_files = [p for p in raw.rglob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]

print(f'CSV files found: {len(csv_files)}')
for c in csv_files[:5]:
    print(' ', c)

print(f'\nImage files found: {len(image_files)}')
print('Sample paths:')
for p in image_files[:5]:
    print(' ', p)

## 6. Build a unified `(image_path, label)` table

Two strategies depending on what we found above:
* **CSV-driven**: read the CSV, expect columns like `IMAGE` and `MEDICINE_NAME` (or similar)
* **Folder-driven**: parent directory name is the class label

The cell below tries the CSV approach first; if no usable CSV exists, it falls back to folder names.

In [ ]:
def _normalize_label(s: str) -> str:
    return str(s).strip().lower()

records = []

csv_loaded = False
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue
    cols_lower = {c.lower(): c for c in df.columns}
    img_col = next((cols_lower[k] for k in cols_lower if 'image' in k or 'file' in k or 'name' in k), None)
    label_col = next((cols_lower[k] for k in cols_lower if 'medicine' in k or 'label' in k or 'word' in k or 'text' in k), None)
    if img_col and label_col and img_col != label_col:
        print(f'Using CSV: {csv_path}\n  image col: {img_col}\n  label col: {label_col}')
        path_lookup = {p.name: p for p in image_files}
        for _, row in df.iterrows():
            img_name = str(row[img_col]).strip()
            label = _normalize_label(row[label_col])
            p = path_lookup.get(img_name) or path_lookup.get(img_name + '.png') or path_lookup.get(img_name + '.jpg')
            if p is not None and label:
                records.append({'image_path': str(p), 'label': label})
        csv_loaded = True
        break

if not csv_loaded:
    print('No usable CSV — falling back to folder name = label')
    for p in image_files:
        records.append({'image_path': str(p), 'label': _normalize_label(p.parent.name)})

df_all = pd.DataFrame(records).drop_duplicates(subset=['image_path']).reset_index(drop=True)
print(f'Total samples: {len(df_all)}')
print(f'Unique labels: {df_all["label"].nunique()}')
df_all.head()

## 7. Quick exploration

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print('Top 20 most common drug names:')
print(df_all['label'].value_counts().head(20))

label_counts = df_all['label'].value_counts()
print(f'\nLabels with only 1 sample: {(label_counts == 1).sum()}')
print(f'Labels with >= 5 samples: {(label_counts >= 5).sum()}')

In [ ]:
sample = df_all.sample(min(12, len(df_all)), random_state=42).reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = Image.open(row['image_path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(row['label'], fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Train / Validation / Test split (80 / 10 / 10)

Stratify by label where each label has ≥ 2 samples; rare labels go to the training set only.

In [ ]:
from sklearn.model_selection import train_test_split

counts = df_all['label'].value_counts()
common = df_all[df_all['label'].isin(counts[counts >= 2].index)].copy()
rare = df_all[df_all['label'].isin(counts[counts < 2].index)].copy()

trainval_common, test_common = train_test_split(
    common, test_size=0.10, random_state=42,
    stratify=common['label'] if common['label'].nunique() > 1 else None,
)
train_common, val_common = train_test_split(
    trainval_common, test_size=0.111, random_state=42,
    stratify=trainval_common['label'] if trainval_common['label'].nunique() > 1 else None,
)

train_df = pd.concat([train_common, rare], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = val_common.reset_index(drop=True)
test_df = test_common.reset_index(drop=True)

print(f'train: {len(train_df)}')
print(f'val  : {len(val_df)}')
print(f'test : {len(test_df)}')

## 9. Save as HuggingFace DatasetDict

This format is what notebook 2 (training) will load directly.

In [ ]:
from datasets import Dataset, DatasetDict, Image as HFImage

def to_hf(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[['image_path', 'label']].rename(columns={'image_path': 'image'}))
    return ds.cast_column('image', HFImage())

splits = DatasetDict({
    'train': to_hf(train_df),
    'validation': to_hf(val_df),
    'test': to_hf(test_df),
})

PREPARED_DIR = f'{DATA_ROOT}/prepared'
splits.save_to_disk(PREPARED_DIR)
print(f'Saved DatasetDict to: {PREPARED_DIR}')
splits

## Done

Move on to **`02_train_trocr.ipynb`** to fine-tune TrOCR on these splits.

**Tip:** the prepared dataset is now in your Drive at `/content/drive/MyDrive/medicare_plus_ocr/data/prepared`. Notebook 2 will pick it up from there automatically.